In [1]:
!pip install pydeck pandas ipywidgets requests

In [2]:
import pydeck as pdk
import pandas as pd
import numpy as np
import ipywidgets as widgets
from IPython.display import display

In [3]:
# Fallback sample (works offline)
df = pd.DataFrame({
    "city": ["Delhi", "Mumbai", "Chennai", "Kolkata", "Bengaluru"],
    "lat": [28.6139, 19.0760, 13.0827, 22.5726, 12.9716],
    "lon": [77.2090, 72.8777, 80.2707, 88.3639, 77.5946],
    "height": [300000, 200000, 250000, 180000, 220000]
})

# If you have a GeoJSON (e.g., buildings), load like:
# import json
# with open("buildings.geojson") as f:
#     gj = json.load(f)
# Then use GeoJsonLayer (see Cell 6)
df

,city,lat,lon,height
0,Delhi,28.6139,77.2090,300000
1,Mumbai,19.0760,72.8777,200000
2,Chennai,13.0827,80.2707,250000
3,Kolkata,22.5726,88.3639,180000
4,Bengaluru,12.9716,77.5946,220000


In [4]:
def height_to_color(h, hmin, hmax):
    t = (h - hmin) / (hmax - hmin + 1e-9)
    # blue → green → red gradient
    r = int(255 * t)
    g = int(255 * (1 - abs(t - 0.5) * 2))
    b = int(255 * (1 - t))
    return [r, g, b]

def add_colors(data):
    hmin, hmax = data["height"].min(), data["height"].max()
    data = data.copy()
    data["color"] = data["height"].apply(lambda h: height_to_color(h, hmin, hmax))
    return data

df_colored = add_colors(df)
df_colored.head()

,city,lat,lon,height,color
0,Delhi,28.6139,77.2090,300000,"[254, 0, 0]"
1,Mumbai,19.0760,72.8777,200000,"[42, 84, 212]"
2,Chennai,13.0827,80.2707,250000,"[148, 212, 106]"
3,Kolkata,22.5726,88.3639,180000,"[0, 0, 255]"
4,Bengaluru,12.9716,77.5946,220000,"[84, 169, 170]"


In [5]:
scale_slider = widgets.FloatSlider(
    value=0.1, min=0.02, max=0.5, step=0.01,
    description="Scale", continuous_update=False
)

style_dropdown = widgets.Dropdown(
    options={
        "Dark": "mapbox://styles/mapbox/dark-v10",
        "Light": "mapbox://styles/mapbox/light-v9",
        "Satellite": "mapbox://styles/mapbox/satellite-v9"
    },
    value="mapbox://styles/mapbox/dark-v10",
    description="Style"
)

display(scale_slider, style_dropdown)

FloatSlider(value=0.1, continuous_update=False, description='Scale', max=0.5, min=0.02, step=0.01)

Dropdown(description='Style', options={'Dark': 'mapbox://styles/mapbox/dark-v10', 'Light': 'mapbox://styles/ma…

In [6]:
def make_deck(data, scale, map_style):
    layer = pdk.Layer(
        "ColumnLayer",
        data,
        get_position='[lon, lat]',
        get_elevation="height",
        elevation_scale=scale,
        radius=40000,
        get_fill_color="color",
        pickable=True,
        auto_highlight=True
    )

    view = pdk.ViewState(latitude=22, longitude=78, zoom=4, pitch=45)

    deck = pdk.Deck(
        layers=[layer],
        initial_view_state=view,
        map_style=map_style,
        tooltip={"text": "{city}\nHeight: {height}"}
    )
    return deck

make_deck(df_colored, scale_slider.value, style_dropdown.value)

{
  "initialViewState": {
    "latitude": 22,
    "longitude": 78,
    "pitch": 45,
    "zoom": 4
  },
  "layers": [
    {
      "@@type": "ColumnLayer",
      "autoHighlight": true,
      "data": [
        {
          "city": "Delhi",
          "color": [
            254,
            0,
            0
          ],
          "height": 300000,
          "lat": 28.6139,
          "lon": 77.209
        },
        {
          "city": "Mumbai",
          "color": [
            42,
            84,
            212
          ],
          "height": 200000,
          "lat": 19.076,
          "lon": 72.8777
        },
        {
          "city": "Chennai",
          "color": [
            148,
            212,
            106
          ],
          "height": 250000,
          "lat": 13.0827,
          "lon": 80.2707
        },
        {
          "city": "Kolkata",
          "color": [
            0,
            0,
            255
          ],
          "height": 180000,
          "lat": 22.5726,
          "lon": 88.3639
        },
        {
          "city": "Bengaluru",
          "color": [
            84,
            169,
            170
          ],
          "height": 220000,
          "lat": 12.9716,
          "lon": 77.5946
        }
      ],
      "elevationScale": 0.1,
      "getElevation": "@@=height",
      "getFillColor": "@@=color",
      "getPosition": "@@=[lon, lat]",
      "id": "d2c9b956-4e5e-47d2-8fef-35cdb811997c",
      "pickable": true,
      "radius": 40000
    }
  ],
  "mapProvider": "carto",
  "mapStyle": "mapbox://styles/mapbox/dark-v10",
  "views": [
    {
      "@@type": "MapView",
      "controller": true
    }
  ]
}

In [7]:
def update(scale, style):
    display(make_deck(df_colored, scale, style))

widgets.interact(update, scale=scale_slider, style=style_dropdown);

interactive(children=(FloatSlider(value=0.1, continuous_update=False, description='Scale', max=0.5, min=0.02, …

In [8]:
# Simulate growth over frames
frames = 30

def animate_growth(frame):
    factor = 0.2 + (frame / frames) * 0.8
    data = df.copy()
    data["height"] = (data["height"] * factor).astype(int)
    data = add_colors(data)
    return make_deck(data, scale_slider.value, style_dropdown.value)

# Play through frames (manual loop for notebook stability)
for f in range(frames):
    display(animate_growth(f))

{
  "initialViewState": {
    "latitude": 22,
    "longitude": 78,
    "pitch": 45,
    "zoom": 4
  },
  "layers": [
    {
      "@@type": "ColumnLayer",
      "autoHighlight": true,
      "data": [
        {
          "city": "Delhi",
          "color": [
            254,
            0,
            0
          ],
          "height": 60000,
          "lat": 28.6139,
          "lon": 77.209
        },
        {
          "city": "Mumbai",
          "color": [
            42,
            84,
            212
          ],
          "height": 40000,
          "lat": 19.076,
          "lon": 72.8777
        },
        {
          "city": "Chennai",
          "color": [
            148,
            212,
            106
          ],
          "height": 50000,
          "lat": 13.0827,
          "lon": 80.2707
        },
        {
          "city": "Kolkata",
          "color": [
            0,
            0,
            255
          ],
          "height": 36000,
          "lat": 22.5726,
          "lon": 88.3639
        },
        {
          "city": "Bengaluru",
          "color": [
            84,
            169,
            170
          ],
          "height": 44000,
          "lat": 12.9716,
          "lon": 77.5946
        }
      ],
      "elevationScale": 0.21,
      "getElevation": "@@=height",
      "getFillColor": "@@=color",
      "getPosition": "@@=[lon, lat]",
      "id": "e3df1eb3-da06-49cd-b09f-c92fdc91ce6f",
      "pickable": true,
      "radius": 40000
    }
  ],
  "mapProvider": "carto",
  "mapStyle": "mapbox://styles/mapbox/dark-v10",
  "views": [
    {
      "@@type": "MapView",
      "controller": true
    }
  ]
}

{
  "initialViewState": {
    "latitude": 22,
    "longitude": 78,
    "pitch": 45,
    "zoom": 4
  },
  "layers": [
    {
      "@@type": "ColumnLayer",
      "autoHighlight": true,
      "data": [
        {
          "city": "Delhi",
          "color": [
            254,
            0,
            0
          ],
          "height": 68000,
          "lat": 28.6139,
          "lon": 77.209
        },
        {
          "city": "Mumbai",
          "color": [
            42,
            84,
            212
          ],
          "height": 45333,
          "lat": 19.076,
          "lon": 72.8777
        },
        {
          "city": "Chennai",
          "color": [
            148,
            212,
            106
          ],
          "height": 56666,
          "lat": 13.0827,
          "lon": 80.2707
        },
        {
          "city": "Kolkata",
          "color": [
            0,
            0,
            255
          ],
          "height": 40800,
          "lat": 22.5726,
          "lon": 88.3639
        },
        {
          "city": "Bengaluru",
          "color": [
            84,
            169,
            170
          ],
          "height": 49866,
          "lat": 12.9716,
          "lon": 77.5946
        }
      ],
      "elevationScale": 0.21,
      "getElevation": "@@=height",
      "getFillColor": "@@=color",
      "getPosition": "@@=[lon, lat]",
      "id": "3a2daddc-e102-4f84-803b-10f7b9ff3666",
      "pickable": true,
      "radius": 40000
    }
  ],
  "mapProvider": "carto",
  "mapStyle": "mapbox://styles/mapbox/dark-v10",
  "views": [
    {
      "@@type": "MapView",
      "controller": true
    }
  ]
}

{
  "initialViewState": {
    "latitude": 22,
    "longitude": 78,
    "pitch": 45,
    "zoom": 4
  },
  "layers": [
    {
      "@@type": "ColumnLayer",
      "autoHighlight": true,
      "data": [
        {
          "city": "Delhi",
          "color": [
            254,
            0,
            0
          ],
          "height": 76000,
          "lat": 28.6139,
          "lon": 77.209
        },
        {
          "city": "Mumbai",
          "color": [
            42,
            84,
            212
          ],
          "height": 50666,
          "lat": 19.076,
          "lon": 72.8777
        },
        {
          "city": "Chennai",
          "color": [
            148,
            212,
            106
          ],
          "height": 63333,
          "lat": 13.0827,
          "lon": 80.2707
        },
        {
          "city": "Kolkata",
          "color": [
            0,
            0,
            255
          ],
          "height": 45600,
          "lat": 22.5726,
          "lon": 88.3639
        },
        {
          "city": "Bengaluru",
          "color": [
            84,
            169,
            170
          ],
          "height": 55733,
          "lat": 12.9716,
          "lon": 77.5946
        }
      ],
      "elevationScale": 0.21,
      "getElevation": "@@=height",
      "getFillColor": "@@=color",
      "getPosition": "@@=[lon, lat]",
      "id": "3fbd19f7-4b90-4eac-8f1a-b34fe98d67c7",
      "pickable": true,
      "radius": 40000
    }
  ],
  "mapProvider": "carto",
  "mapStyle": "mapbox://styles/mapbox/dark-v10",
  "views": [
    {
      "@@type": "MapView",
      "controller": true
    }
  ]
}

{
  "initialViewState": {
    "latitude": 22,
    "longitude": 78,
    "pitch": 45,
    "zoom": 4
  },
  "layers": [
    {
      "@@type": "ColumnLayer",
      "autoHighlight": true,
      "data": [
        {
          "city": "Delhi",
          "color": [
            254,
            0,
            0
          ],
          "height": 84000,
          "lat": 28.6139,
          "lon": 77.209
        },
        {
          "city": "Mumbai",
          "color": [
            42,
            84,
            212
          ],
          "height": 56000,
          "lat": 19.076,
          "lon": 72.8777
        },
        {
          "city": "Chennai",
          "color": [
            148,
            212,
            106
          ],
          "height": 70000,
          "lat": 13.0827,
          "lon": 80.2707
        },
        {
          "city": "Kolkata",
          "color": [
            0,
            0,
            255
          ],
          "height": 50400,
          "lat": 22.5726,
          "lon": 88.3639
        },
        {
          "city": "Bengaluru",
          "color": [
            84,
            169,
            170
          ],
          "height": 61600,
          "lat": 12.9716,
          "lon": 77.5946
        }
      ],
      "elevationScale": 0.21,
      "getElevation": "@@=height",
      "getFillColor": "@@=color",
      "getPosition": "@@=[lon, lat]",
      "id": "1a950e2d-a3c0-4335-bb73-64684e892203",
      "pickable": true,
      "radius": 40000
    }
  ],
  "mapProvider": "carto",
  "mapStyle": "mapbox://styles/mapbox/dark-v10",
  "views": [
    {
      "@@type": "MapView",
      "controller": true
    }
  ]
}

{
  "initialViewState": {
    "latitude": 22,
    "longitude": 78,
    "pitch": 45,
    "zoom": 4
  },
  "layers": [
    {
      "@@type": "ColumnLayer",
      "autoHighlight": true,
      "data": [
        {
          "city": "Delhi",
          "color": [
            254,
            0,
            0
          ],
          "height": 92000,
          "lat": 28.6139,
          "lon": 77.209
        },
        {
          "city": "Mumbai",
          "color": [
            42,
            84,
            212
          ],
          "height": 61333,
          "lat": 19.076,
          "lon": 72.8777
        },
        {
          "city": "Chennai",
          "color": [
            148,
            212,
            106
          ],
          "height": 76666,
          "lat": 13.0827,
          "lon": 80.2707
        },
        {
          "city": "Kolkata",
          "color": [
            0,
            0,
            255
          ],
          "height": 55200,
          "lat": 22.5726,
          "lon": 88.3639
        },
        {
          "city": "Bengaluru",
          "color": [
            84,
            169,
            170
          ],
          "height": 67466,
          "lat": 12.9716,
          "lon": 77.5946
        }
      ],
      "elevationScale": 0.21,
      "getElevation": "@@=height",
      "getFillColor": "@@=color",
      "getPosition": "@@=[lon, lat]",
      "id": "56b20c48-0314-4aac-908a-cd2beb83986d",
      "pickable": true,
      "radius": 40000
    }
  ],
  "mapProvider": "carto",
  "mapStyle": "mapbox://styles/mapbox/dark-v10",
  "views": [
    {
      "@@type": "MapView",
      "controller": true
    }
  ]
}

...

{
  "initialViewState": {
    "latitude": 22,
    "longitude": 78,
    "pitch": 45,
    "zoom": 4
  },
  "layers": [
    {
      "@@type": "ColumnLayer",
      "autoHighlight": true,
      "data": [
        {
          "city": "Delhi",
          "color": [
            254,
            0,
            0
          ],
          "height": 260000,
          "lat": 28.6139,
          "lon": 77.209
        },
        {
          "city": "Mumbai",
          "color": [
            42,
            84,
            212
          ],
          "height": 173333,
          "lat": 19.076,
          "lon": 72.8777
        },
        {
          "city": "Chennai",
          "color": [
            148,
            212,
            106
          ],
          "height": 216666,
          "lat": 13.0827,
          "lon": 80.2707
        },
        {
          "city": "Kolkata",
          "color": [
            0,
            0,
            255
          ],
          "height": 156000,
          "lat": 22.5726,
          "lon": 88.3639
        },
        {
          "city": "Bengaluru",
          "color": [
            84,
            169,
            170
          ],
          "height": 190666,
          "lat": 12.9716,
          "lon": 77.5946
        }
      ],
      "elevationScale": 0.21,
      "getElevation": "@@=height",
      "getFillColor": "@@=color",
      "getPosition": "@@=[lon, lat]",
      "id": "e82458f9-95c7-4296-9394-57b7d865b01b",
      "pickable": true,
      "radius": 40000
    }
  ],
  "mapProvider": "carto",
  "mapStyle": "mapbox://styles/mapbox/dark-v10",
  "views": [
    {
      "@@type": "MapView",
      "controller": true
    }
  ]
}

{
  "initialViewState": {
    "latitude": 22,
    "longitude": 78,
    "pitch": 45,
    "zoom": 4
  },
  "layers": [
    {
      "@@type": "ColumnLayer",
      "autoHighlight": true,
      "data": [
        {
          "city": "Delhi",
          "color": [
            254,
            0,
            0
          ],
          "height": 268000,
          "lat": 28.6139,
          "lon": 77.209
        },
        {
          "city": "Mumbai",
          "color": [
            42,
            84,
            212
          ],
          "height": 178666,
          "lat": 19.076,
          "lon": 72.8777
        },
        {
          "city": "Chennai",
          "color": [
            148,
            212,
            106
          ],
          "height": 223333,
          "lat": 13.0827,
          "lon": 80.2707
        },
        {
          "city": "Kolkata",
          "color": [
            0,
            0,
            255
          ],
          "height": 160800,
          "lat": 22.5726,
          "lon": 88.3639
        },
        {
          "city": "Bengaluru",
          "color": [
            84,
            169,
            170
          ],
          "height": 196533,
          "lat": 12.9716,
          "lon": 77.5946
        }
      ],
      "elevationScale": 0.21,
      "getElevation": "@@=height",
      "getFillColor": "@@=color",
      "getPosition": "@@=[lon, lat]",
      "id": "2f3acc14-ae48-4317-875a-f60bd317dc44",
      "pickable": true,
      "radius": 40000
    }
  ],
  "mapProvider": "carto",
  "mapStyle": "mapbox://styles/mapbox/dark-v10",
  "views": [
    {
      "@@type": "MapView",
      "controller": true
    }
  ]
}

{
  "initialViewState": {
    "latitude": 22,
    "longitude": 78,
    "pitch": 45,
    "zoom": 4
  },
  "layers": [
    {
      "@@type": "ColumnLayer",
      "autoHighlight": true,
      "data": [
        {
          "city": "Delhi",
          "color": [
            254,
            0,
            0
          ],
          "height": 276000,
          "lat": 28.6139,
          "lon": 77.209
        },
        {
          "city": "Mumbai",
          "color": [
            42,
            84,
            212
          ],
          "height": 184000,
          "lat": 19.076,
          "lon": 72.8777
        },
        {
          "city": "Chennai",
          "color": [
            148,
            212,
            106
          ],
          "height": 230000,
          "lat": 13.0827,
          "lon": 80.2707
        },
        {
          "city": "Kolkata",
          "color": [
            0,
            0,
            255
          ],
          "height": 165600,
          "lat": 22.5726,
          "lon": 88.3639
        },
        {
          "city": "Bengaluru",
          "color": [
            84,
            169,
            170
          ],
          "height": 202400,
          "lat": 12.9716,
          "lon": 77.5946
        }
      ],
      "elevationScale": 0.21,
      "getElevation": "@@=height",
      "getFillColor": "@@=color",
      "getPosition": "@@=[lon, lat]",
      "id": "2fdf8b2e-be14-45e3-979e-0470aa84b661",
      "pickable": true,
      "radius": 40000
    }
  ],
  "mapProvider": "carto",
  "mapStyle": "mapbox://styles/mapbox/dark-v10",
  "views": [
    {
      "@@type": "MapView",
      "controller": true
    }
  ]
}

{
  "initialViewState": {
    "latitude": 22,
    "longitude": 78,
    "pitch": 45,
    "zoom": 4
  },
  "layers": [
    {
      "@@type": "ColumnLayer",
      "autoHighlight": true,
      "data": [
        {
          "city": "Delhi",
          "color": [
            254,
            0,
            0
          ],
          "height": 284000,
          "lat": 28.6139,
          "lon": 77.209
        },
        {
          "city": "Mumbai",
          "color": [
            42,
            84,
            212
          ],
          "height": 189333,
          "lat": 19.076,
          "lon": 72.8777
        },
        {
          "city": "Chennai",
          "color": [
            148,
            212,
            106
          ],
          "height": 236666,
          "lat": 13.0827,
          "lon": 80.2707
        },
        {
          "city": "Kolkata",
          "color": [
            0,
            0,
            255
          ],
          "height": 170400,
          "lat": 22.5726,
          "lon": 88.3639
        },
        {
          "city": "Bengaluru",
          "color": [
            84,
            169,
            170
          ],
          "height": 208266,
          "lat": 12.9716,
          "lon": 77.5946
        }
      ],
      "elevationScale": 0.21,
      "getElevation": "@@=height",
      "getFillColor": "@@=color",
      "getPosition": "@@=[lon, lat]",
      "id": "6404aeaf-695c-4271-8547-ace2e8ab5750",
      "pickable": true,
      "radius": 40000
    }
  ],
  "mapProvider": "carto",
  "mapStyle": "mapbox://styles/mapbox/dark-v10",
  "views": [
    {
      "@@type": "MapView",
      "controller": true
    }
  ]
}

{
  "initialViewState": {
    "latitude": 22,
    "longitude": 78,
    "pitch": 45,
    "zoom": 4
  },
  "layers": [
    {
      "@@type": "ColumnLayer",
      "autoHighlight": true,
      "data": [
        {
          "city": "Delhi",
          "color": [
            254,
            0,
            0
          ],
          "height": 292000,
          "lat": 28.6139,
          "lon": 77.209
        },
        {
          "city": "Mumbai",
          "color": [
            42,
            84,
            212
          ],
          "height": 194666,
          "lat": 19.076,
          "lon": 72.8777
        },
        {
          "city": "Chennai",
          "color": [
            148,
            212,
            106
          ],
          "height": 243333,
          "lat": 13.0827,
          "lon": 80.2707
        },
        {
          "city": "Kolkata",
          "color": [
            0,
            0,
            255
          ],
          "height": 175200,
          "lat": 22.5726,
          "lon": 88.3639
        },
        {
          "city": "Bengaluru",
          "color": [
            84,
            169,
            170
          ],
          "height": 214133,
          "lat": 12.9716,
          "lon": 77.5946
        }
      ],
      "elevationScale": 0.21,
      "getElevation": "@@=height",
      "getFillColor": "@@=color",
      "getPosition": "@@=[lon, lat]",
      "id": "cb0fb239-a4a7-4956-88e4-f57f9338325c",
      "pickable": true,
      "radius": 40000
    }
  ],
  "mapProvider": "carto",
  "mapStyle": "mapbox://styles/mapbox/dark-v10",
  "views": [
    {
      "@@type": "MapView",
      "controller": true
    }
  ]
}

In [9]:
# If you have a buildings GeoJSON (with 'height' property):
# import json
# with open("buildings.geojson") as f:
#     gj = json.load(f)

# gj_layer = pdk.Layer(
#     "GeoJsonLayer",
#     gj,
#     extruded=True,
#     get_elevation="properties.height",
#     get_fill_color=[200, 100, 240],
#     pickable=True
# )

# pdk.Deck(
#     layers=[gj_layer],
#     initial_view_state=pdk.ViewState(latitude=22, longitude=78, zoom=12, pitch=45),
#     map_style=style_dropdown.value
# )